# Block Model Estimation — Grade Assignment and Product Classification

**Objective:** Estimate geochemical grades (Fe, SiO₂, G1, G2, G3) for each block in the model using length-weighted averages from the drill hole assays, then classify each block into an ore product type.

**Methodology:**
1. Assign each sample to its corresponding block using grid indices
2. Compute length-weighted average grades per block
3. Classify blocks into product categories (Lump Ore, Sinter Feed, Pellet Feed, etc.)
4. Analyze results and export the estimated block model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import floor
import os

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = os.path.join("..", "data")
GEOCHEM_COLS = ["FE", "SI", "G1", "G2", "G3"]

## 1. Load Cleaned Data

In [ ]:
assays = pd.read_csv(os.path.join(DATA_DIR, "assays_cleaned.csv"))
block_model = pd.read_csv(os.path.join(DATA_DIR, "block_model_cleaned.csv"))

print(f"Assays:      {assays.shape[0]:,} samples")
print(f"Block Model: {block_model.shape[0]:,} blocks")

## 2. Define Block Grid and Assign Samples

The block model uses a regular grid with:
- **Block size:** 50 × 50 × 25 m (X × Y × Z)

Each sample is assigned to its corresponding block using integer grid indices `(i, j, k)` based on its center coordinates.

In [ ]:
# Grid origin from block model
x0 = block_model["X"].min()
y0 = block_model["Y"].min()
z0 = block_model["Z"].min()

# Block dimensions
DX, DY, DZ = 50, 50, 25

print(f"Grid origin: ({x0:,.0f}, {y0:,.0f}, {z0:,.0f})")
print(f"Block size:  {DX} × {DY} × {DZ} m")

# Assign grid indices to samples
assays["i"] = ((assays["X_center"] - x0) / DX).apply(floor).astype("int32")
assays["j"] = ((assays["Y_center"] - y0) / DY).apply(floor).astype("int32")
assays["k"] = ((assays["Z_center"] - z0) / DZ).apply(floor).astype("int32")

# Assign grid indices to block model
block_model["i"] = ((block_model["X"] - x0) / DX).apply(floor).astype("int32")
block_model["j"] = ((block_model["Y"] - y0) / DY).apply(floor).astype("int32")
block_model["k"] = ((block_model["Z"] - z0) / DZ).apply(floor).astype("int32")

print(f"\nSamples assigned to {assays.groupby(['i','j','k']).ngroups:,} unique blocks")

## 3. Length-Weighted Average Estimation

For each block, we compute the weighted average of each geochemical variable, using the sample interval length as the weight. This accounts for the fact that longer intervals represent a greater volume of material.

$$\bar{x}_{block} = \frac{\sum_{i=1}^{n} x_i \cdot L_i}{\sum_{i=1}^{n} L_i}$$

Where $x_i$ is the grade and $L_i$ is the sample length.

In [ ]:
def weighted_mean(group):
    """Compute length-weighted average for each geochemical variable."""
    results = {}
    for var in GEOCHEM_COLS:
        valid = group.dropna(subset=[var])
        if len(valid) > 0 and valid["LENGTH"].sum() > 0:
            results[var] = np.average(valid[var], weights=valid["LENGTH"])
        else:
            results[var] = np.nan
    return pd.Series(results)

# Compute estimates per block
estimates = (
    assays.groupby(["i", "j", "k"])
    .apply(weighted_mean, include_groups=False)
    .reset_index()
)

print(f"Estimated grades for {len(estimates):,} blocks")
estimates.head(10)

## 4. Merge Estimates into Block Model

In [ ]:
# Drop original grade columns from block model (they contain mostly NaN/-99)
block_model = block_model.drop(columns=GEOCHEM_COLS, errors="ignore")

# Merge estimated grades
block_model = block_model.merge(estimates, on=["i", "j", "k"], how="left")

estimated_count = block_model["FE"].notna().sum()
total_blocks = len(block_model)

print(f"Blocks with estimated grades: {estimated_count:,} / {total_blocks:,} "
      f"({estimated_count/total_blocks*100:.1f}%)")
print(f"Blocks without data (no nearby samples): {total_blocks - estimated_count:,}")

block_model.head(10)

## 5. Product Classification

Each block is classified into an ore product based on its estimated grades and granulometry:

| Product | Criteria |
|---------|----------|
| **Lump Ore** | G1 ≥ 60%, Fe > 60%, Si < 5% |
| **Sinter Feed** | G2 ≥ 50%, Fe ≥ 56%, Si ≤ 10% |
| **Pellet Feed** | G3 ≥ 50%, Fe ≥ 56%, Si ≤ 8% |
| **Mixed / Blending Required** | 50% ≤ Fe ≤ 60%, 5% ≤ Si ≤ 15% |
| **Byproduct / Waste** | Fe < 50% or Si > 15% |

In [ ]:
def classify_product(row):
    """Classify a block into an ore product based on grade and granulometry."""
    fe = row["FE"]
    si = row["SI"]
    g1, g2, g3 = row["G1"], row["G2"], row["G3"]
    lito = str(row.get("LITOTIPO", "")).upper()

    # Skip blocks without grade data
    if pd.isna(fe) or pd.isna(si):
        return "UNCLASSIFIED"

    # Primary classification by granulometry + grade
    if pd.notna(g1) and g1 >= 60 and fe > 60 and si < 5:
        return "LUMP ORE"
    if pd.notna(g2) and g2 >= 50 and fe >= 56 and si <= 10:
        return "SINTER FEED"
    if pd.notna(g3) and g3 >= 50 and fe >= 56 and si <= 8:
        return "PELLET FEED"

    # Fallback classification by lithotype
    if lito == "HF":
        return "SINTER FEED / PELLET FEED"
    if lito == "HC":
        return "SINTER FEED"
    if lito == "CM":
        return "PELLET FEED"

    # Grade-only classification
    if fe > 60 and si < 5:
        return "SINTER FEED / PELLET FEED"
    if 50 <= fe <= 60 and 5 <= si <= 15:
        return "MIXED / BLENDING"
    if fe < 50 or si > 15:
        return "BYPRODUCT / WASTE"

    return "UNCLASSIFIED"

block_model["PRODUCT"] = block_model.apply(classify_product, axis=1)

print("Product classification results:")
print(block_model["PRODUCT"].value_counts().to_string())

## 6. Results Visualization

In [ ]:
# Product distribution pie chart
product_counts = block_model["PRODUCT"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Pie chart
colors_map = {
    "LUMP ORE": "#2ecc71",
    "SINTER FEED": "#3498db",
    "PELLET FEED": "#9b59b6",
    "SINTER FEED / PELLET FEED": "#1abc9c",
    "MIXED / BLENDING": "#f39c12",
    "BYPRODUCT / WASTE": "#e74c3c",
    "UNCLASSIFIED": "#95a5a6",
}
colors = [colors_map.get(p, "#bdc3c7") for p in product_counts.index]

axes[0].pie(product_counts.values, labels=product_counts.index, autopct="%1.1f%%",
            colors=colors, startangle=140, textprops={"fontsize": 9})
axes[0].set_title("Block Model — Product Distribution", fontsize=13)

# Bar chart
bars = axes[1].barh(product_counts.index, product_counts.values, color=colors, edgecolor="white")
axes[1].set_xlabel("Number of Blocks")
axes[1].set_title("Block Count by Product Type", fontsize=13)
axes[1].bar_label(bars, padding=3, fontsize=9)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### 6.1 Estimated Fe Grade — Plan View

In [ ]:
estimated = block_model.dropna(subset=["FE"])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plan view — Fe
sc1 = axes[0].scatter(estimated["X"], estimated["Y"], c=estimated["FE"],
                      cmap="RdYlGn", s=12, alpha=0.7, edgecolors="none")
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")
axes[0].set_title("Estimated Fe (%) — Plan View")
axes[0].ticklabel_format(style="plain")
plt.colorbar(sc1, ax=axes[0], label="Fe (%)")

# Cross-section — Fe
sc2 = axes[1].scatter(estimated["X"], estimated["Z"], c=estimated["FE"],
                      cmap="RdYlGn", s=12, alpha=0.7, edgecolors="none")
axes[1].set_xlabel("Easting (m)")
axes[1].set_ylabel("Elevation (m)")
axes[1].set_title("Estimated Fe (%) — Cross Section")
plt.colorbar(sc2, ax=axes[1], label="Fe (%)")

plt.tight_layout()
plt.show()

### 6.2 Estimated Grade Statistics

In [ ]:
print("Estimated Block Model — Grade Statistics")
print("=" * 50)
block_model[GEOCHEM_COLS].describe().round(2)

### 6.3 Fe Distribution — Before vs After Estimation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Sample Fe (from assays)
fe_samples = assays["FE"].dropna()
ax.hist(fe_samples, bins=40, alpha=0.5, color="#c0392b", edgecolor="white",
        label=f"Samples (n={len(fe_samples):,}, mean={fe_samples.mean():.1f}%)", density=True)

# Estimated block Fe
fe_blocks = block_model["FE"].dropna()
ax.hist(fe_blocks, bins=40, alpha=0.5, color="#2980b9", edgecolor="white",
        label=f"Block Estimates (n={len(fe_blocks):,}, mean={fe_blocks.mean():.1f}%)", density=True)

ax.set_xlabel("Fe (%)")
ax.set_ylabel("Density")
ax.set_title("Fe Grade Distribution — Samples vs Block Estimates")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Export Final Block Model

In [ ]:
output_path = os.path.join(DATA_DIR, "block_model_estimated.csv")
block_model.to_csv(output_path, index=False)

print(f"Final block model exported to: {output_path}")
print(f"Total blocks: {len(block_model):,}")
print(f"Blocks with estimates: {block_model['FE'].notna().sum():,}")
print(f"\nColumns: {block_model.columns.tolist()}")

## Summary

**Block Model Estimation Pipeline:**

1. Loaded cleaned drill hole assays with 3D sample coordinates and interval lengths
2. Assigned each sample to its corresponding block via grid indices (50×50×25 m blocks)
3. Computed **length-weighted average grades** (Fe, SiO₂, G1, G2, G3) per block
4. Classified blocks into ore products: Lump Ore, Sinter Feed, Pellet Feed, Mixed, or Waste
5. Exported the final estimated and classified block model

**Key observations:**
- Block estimates show a smoother distribution than raw samples (expected averaging effect)
- Product classification follows Vale's standard iron ore product specifications
- Blocks without nearby drill hole samples remain unclassified — these would require additional drilling or interpolation techniques (e.g., Kriging) for estimation